# Results Section - ICWSM 15.09.2026 Submission

Consolidating the results that would be interesting and worth reporting in the paper for submission.
We ask the following research questions.
1. How prevalent is racialised reviews? - To answer this we look at the racialised rate over time and space
2. Are there significant connections betwen the racialised rate and socio economic and political factors
3. Who is leaving these reviews and how different are their concerns

# Setup 

Load the two canonical inputs and do a first-touch shape/label sanity check against the counts documented in the corpus README (7,768 racialised, 504,072 social=yes, 1,265,806 total) — if these don't match, something upstream has drifted and everything after this point is suspect.


In [ ]:
#packages
import warnings
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import geopandas as gpd

import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest


warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

from statsmodels.stats.proportion import proportion_confint

def slugify(title):
    """Turn a plot title into a filesystem-safe filename stem (used when saving figures)."""
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:80]

In [ ]:
# Wilson score interval helper, used throughout Part 1 (WHEN's yearly/monthly rates, WHERE's
# constituency/state prevalence) -- defined once here, before either section needs it.
from statsmodels.stats.proportion import proportion_confint

def wilson(successes, n):
    if n == 0:
        return np.nan, np.nan, np.nan
    rate = successes / n
    lo, hi = proportion_confint(successes, n, method="wilson")
    return rate, lo, hi

# Data

In [ ]:
ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
FIGURES_DIR = ANALYSIS_DIR / "descriptive_analysis_figures"
FIGURES_DIR.mkdir(exist_ok=True)

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
import sys
sys.path.insert(0, str(SOCIO_DIR))
from merge_political_data import POLITICAL_COLUMNS, load_political_rows

# party shares (CDU/CSU merged), not raw per-party vote counts -- reuses the same
# cleaning merge_political_data.py already does for federal_baseline_eng_selected_political.csv
political_by_constituency = load_political_rows(SOCIO_DIR / "votes_share_raw.csv")
political = pd.DataFrame.from_dict(political_by_constituency, orient="index")
political.index = political.index.astype(int)
political.index.name = "constituency_code"
political = political.reset_index()
for col in POLITICAL_COLUMNS:
    if col != "direct_mandate_party":
        political[col] = pd.to_numeric(political[col], errors="coerce")
socio = pd.read_csv(SOCIO_DIR / "socioeconomic_eng_clean.csv")

print(f"corpus: {corpus.shape[0]:,} rows x {corpus.shape[1]} cols")
print(f"political: {political.shape[0]} constituencies x {political.shape[1]} cols")
print(f"socioeconomic: {socio.shape[0]} constituencies x {socio.shape[1]} cols")


assert corpus.shape[0] == 1_265_806, "row count drifted from documented corpus size"
assert (corpus["racialised"] == "yes").sum() == 7_768, "racialised=yes count drifted"
assert (corpus["social"] == "yes").sum() == 504_072, "social=yes count drifted"
assert political.shape[0] == 299, "expected exactly 299 Bundestag constituencies"
assert socio.shape[0] == 299, "expected exactly 299 Bundestag constituencies"
print("Sanity checks passed: corpus and political data match documented provenance.")



## Corpus data

In [ ]:
# racialised=yes subset, reused throughout Part 1 (WHEN's dimension-over-time facet,
# WHERE's pool-concentration checks, WHO's ethnicity/language facet, and the
# dimension-content-overview section) -- built once here rather than re-derived per section.
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()
print(f"racialised=yes reviews: {len(racialised_df):,}")

#breakdown of the  racilised=yes coverage by dimensions
DIMENSIONS = ["approach", "theme", "connotation", "context"]

def is_classified(cell):
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return len(cell) > 0
    if isinstance(cell, str):
        return len(cell) > 0
    return False  # NaN or unexpected type

coverage_rows = []
label_rows = []
for dim in DIMENSIONS:
    n_classified = racialised_df[dim].apply(is_classified).sum()
    coverage_rows.append({
        "dimension": dim,
        "n_classified": n_classified,
        "pct_classified": n_classified / len(racialised_df),
    })
    counts = racialised_df[dim].explode().value_counts()
    for label, n in counts.items():
        label_rows.append({
            "dimension": dim,
            "label": label,
            "n_reviews": n,
            "pct_of_racialised_reviews": n / len(racialised_df),
        })

coverage_table = pd.DataFrame(coverage_rows).set_index("dimension")
print(f"Classification coverage (of {len(racialised_df):,} racialised reviews):")
print(coverage_table.assign(pct_classified=lambda d: (d["pct_classified"] * 100).round(1)))
assert (coverage_table["pct_classified"] == 1.0).all(), "not every racialised review is classified on every dimension"

label_breakdown = pd.DataFrame(label_rows).set_index(["dimension", "label"])
label_breakdown["pct_of_racialised_reviews"] = (label_breakdown["pct_of_racialised_reviews"] * 100).round(1)
display(label_breakdown)

###  place → constituency spatial join


 *method* — dissolve the 299 constituency polygons into one Germany polygon, keep only points strictly inside it, then `sjoin_nearest` the kept points against the individual polygons so every retained point is guaranteed exactly one constituency, even ones that sit awkwardly on a shared border. 
We use that method  once at the **place level** (5,529 unique places, keyed by `unique_identifier`) and then broadcast the result back onto all 1,265,806 review rows — same logic, ~230x fewer spatial operations, and verified end-to-end against the current canonical corpus.

**CRS choice:** the shapefile's native CRS is `EPSG:25832` (ETRS89 / UTM zone 32N), a metric projection appropriate for Germany. We reproject points into that CRS — reprojecting to the shapefile's own native CRS avoids an extra, unnecessary reprojection step and the distortion Web Mercator introduces at this latitude.

In [ ]:
# --- place-level table (one row per unique_identifier) ---
places = (
    corpus[["unique_identifier", "place_name", "latitude", "longitude"]]
    .drop_duplicates(subset="unique_identifier")
    .reset_index(drop=True)
)
n_missing_coords = places[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"places (unique unique_identifier): {len(places):,}")
print(f"places missing lat/lon: {n_missing_coords}")

#convert the places dataframe to a GeoDataFrame with geometry column
# and crs set to EPSG:4326, then reproject to EPSG:25832
places_geo = gpd.GeoDataFrame(
    places,
    geometry=gpd.points_from_xy(places["longitude"], places["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")

# --- constituency shapefile ---
constituencies = gpd.read_file(SHAPEFILE_PATH)
print(f"constituencies: {len(constituencies)} polygons, CRS = {constituencies.crs}")

n_invalid = (~constituencies.geometry.is_valid).sum()
print(f"invalid polygon geometries before repair: {n_invalid}")
if n_invalid:
    constituencies["geometry"] = constituencies.geometry.buffer(0)

assert constituencies.crs.to_string() == "EPSG:25832", "shapefile CRS is not what we expect"
assert constituencies["WKR_NR"].nunique() == 299, "expected 299 unique constituency numbers"

# --- step 1: keep only places strictly inside Germany (dissolved boundary) ---
germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)

inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()
n_dropped = len(places_geo) - len(places_inside)
print(f"places strictly inside Germany polygon: {len(places_inside):,} (dropped as outside: {n_dropped})")

#--- step 2: assign each inside place to its nearest constituency polygon ---
# (guarantees a match even for a point sitting exactly on / just outside a shared border)
joined = gpd.sjoin_nearest(
    places_inside,
    constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]],
    how="left",
)

n_dupe_matches = joined.duplicated(subset="unique_identifier").sum()
print(f"places matched more than once (nearest-neighbour ties): {n_dupe_matches}")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")

place_constituency = joined.rename(
    columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"}
)[["unique_identifier", "constituency_code", "constituency", "state"]].reset_index(drop=True)

print(f"final place -> constituency mapping: {len(place_constituency):,} rows")
assert place_constituency["unique_identifier"].is_unique, "join must be 1:1 on unique_identifier"

# --- broadcast the place-level mapping back onto every review row ---
corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

n_reviews_unmatched = corpus["constituency_code"].isna().sum()
n_places_unmatched = places["unique_identifier"].nunique() - place_constituency["unique_identifier"].nunique()
print(f"review rows without a constituency match: {n_reviews_unmatched:,} ({n_reviews_unmatched / len(corpus):.4%})")
print(f"places without a constituency match: {n_places_unmatched:,}")

if n_reviews_unmatched:
    print("\nplace_name of unmatched places:")
    print(
        corpus.loc[corpus["constituency_code"].isna(), "place_name"].drop_duplicates().head(10).to_list()
    )

east_states = {
    "Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen",
}
corpus["region"] = np.where(corpus["state"].isin(east_states), "East", "West")
corpus.loc[corpus["state"] == "Berlin", "region"] = "Berlin"

print("\nreviews by region (East / West / Berlin split out):")
print(corpus["region"].value_counts(dropna=False))


## Socio-economic & Political Data

In [ ]:
political.info()

In [ ]:
socio_political = socio.merge(political, on="constituency_code", how="left")
assert socio_political["valid_votes"].notna().all(), "missing political data for some constituency"
socio_political.shape

In [ ]:
socio_political.to_csv(SOCIO_DIR / "socio_political_data.csv", index=False)

**Note** A complete match: all 5,528 places with reviews in `master_corpus_final` (5,529 verified POIs minus the one documented in the `gmaps_dat/README` as having no reviews) fall inside Germany and get exactly one constituency, with zero drops and zero nearest-neighbour ties — so **100% of the 1,265,806 review rows now carry a `constituency_code`/`constituency`/`state`**. We also derive a simple `region` (East/West/Berlin) split for later comparisons, reusing the standard German reunification state grouping.

### The racilaised rate
There could be two ways to look at the racialised rate:
1. racialised rate as the racialised reviews/all reviews for a constituency (e.g. how much racialised content a random visitor is likely to encounter across all reviews.)
2. racialised rate as the racialsed reviews/social ravies in a constituency(e.g. given someone commented on people/atmosphere/behavior at all, how likely was it to be racialised.)

The second method is of more interest because before looking at racialisation, we first interested in the propensity to discuss the social experience in a review, this we can consider a proxy for social mixing. Then we are interest in how often there is a propensit yot racialise when sharing the social experience


# 1. How prevalent is racialised reviews? 

## To answer this we look at the racialised rate over time 
Some questions
1. Is racialised discourse increasing over time?
2. Does the racialised rate vary per seson?

In [ ]:
#check reviews without timestamps
has_ts = corpus["review_timestamp"].notna()
print(f"reviews with a parsed timestamp: {has_ts.sum():,} / {len(corpus):,} ({has_ts.mean():.1%})")

# missingness by racialised status (restricted to social=yes, where racialised is defined)
social_mask = corpus["social"] == "yes"
miss_by_racialised = (
    corpus.loc[social_mask]
    .assign(has_ts=has_ts[social_mask])
    .groupby("racialised")["has_ts"]
    .mean()
)
print("\ntimestamp coverage by racialised status (within social=yes):")
print(miss_by_racialised)

# missingness by state
miss_by_state = corpus.assign(has_ts=has_ts).groupby("state")["has_ts"].mean().sort_values()
print("\ntimestamp coverage by state (min/max):")
print(miss_by_state.head(3))
print("...")
print(miss_by_state.tail(3))


In [ ]:
REL_TIME_PATTERN = re.compile(
    r"^(?:Edited\s+)?(a|an|\d+)\s+(year|month|week|day|hour|minute)s?\s+ago$", re.IGNORECASE
)
UNIT_TO_DAYS = {"year": 365.25, "month": 30.44, "week": 7, "day": 1, "hour": 1 / 24, "minute": 1 / (24 * 60)}

def parse_relative_time(text):
    m = REL_TIME_PATTERN.match(text) if isinstance(text, str) else None
    if not m:
        return None
    qty_raw, unit = m.groups()
    qty = 1 if qty_raw.lower() in ("a", "an") else int(qty_raw)
    return pd.Timedelta(days=qty * UNIT_TO_DAYS[unit.lower()])

missing_ts_mask = corpus["review_timestamp"].isna()
deltas = corpus.loc[missing_ts_mask, "review_time"].apply(parse_relative_time)
n_parsed = deltas.notna().sum()
print(f"missing timestamps: {missing_ts_mask.sum():,}; recoverable via relative-time text: {n_parsed:,} "
      f"({n_parsed / missing_ts_mask.sum():.1%})")

# review_timestamp is tz-aware (parsed from the gmaps_poi DB); place_retrieved_time is tz-naive and the
# estimate ends up nanosecond-precision. We only need year/month granularity for the trend, so build
# both sides as plain tz-naive 'us'-precision series and combine, rather than assigning in place.
base = corpus["review_timestamp"].dt.tz_localize(None).astype("datetime64[us]")
estimated = pd.Series(pd.NaT, index=corpus.index, dtype="datetime64[us]")
raw_est = (corpus.loc[missing_ts_mask, "retrieved_time"] - deltas).astype("datetime64[us]")
#if there are missing use the place_retrieved_time - parsed delta to estimate the review_timestamp
raw_est = (corpus.loc[missing_ts_mask, "place_retrieved_time"] - deltas).astype("datetime64[us]")
estimated.loc[missing_ts_mask] = raw_est

corpus["review_timestamp_est"] = base.fillna(estimated)
corpus["timestamp_is_estimated"] = missing_ts_mask & corpus["review_timestamp_est"].notna()

new_coverage = corpus["review_timestamp_est"].notna().mean()
print(f"timestamp coverage after recovery: {new_coverage:.2%} (was {has_ts.mean():.1%})")
print(f"still unrecoverable (no review_time text at all): {corpus['review_timestamp_est'].isna().sum():,}")

### 

In [ ]:
# --- pool pre-2016 years into a single bucket (too sparse to read year-by-year), then plot
# racialised rate (line, top) against review volume -- all reviews vs social=yes reviews (bars, bottom) ---

CUTOFF_YEAR = 2016

def make_bucket(year):
    return f"pre-{CUTOFF_YEAR}" if year < CUTOFF_YEAR else str(int(year))

def bucketed_rate_table(df, count_col, success_fn):
    d = df[df["review_year"] < 2026].copy()  # drop partial current year
    d["bucket"] = d["review_year"].apply(make_bucket)
    table = (
        d.groupby("bucket")
        .agg(n_reviews=(count_col, "size"), n_success=(count_col, success_fn))
    )
    order = [f"pre-{CUTOFF_YEAR}"] + [str(y) for y in range(CUTOFF_YEAR, 2026)]
    table = table.reindex(order).dropna()
    rates = table.apply(
        lambda r: pd.Series(wilson(r["n_success"], r["n_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
        axis=1,
    )
    return table.join(rates)


# n_reviews here = all reviews per bucket, rate = social rate (not plotted, kept for volume bars)
corpus["review_year"] = corpus["review_timestamp_est"].dt.year
social_mask = corpus["social"] == "yes"
yearly_totals = bucketed_rate_table(corpus, "social", lambda s: (s == "yes").sum())

# n_reviews here = social reviews per bucket, rate = racialised rate
ts_social = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_social["review_year"] = ts_social["review_timestamp_est"].dt.year
yearly_social = bucketed_rate_table(ts_social, "racialised", lambda s: (s == "yes").sum())

buckets = yearly_totals.index
x = list(range(len(buckets)))
width = 0.4
social_aligned = yearly_social.reindex(buckets)


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})

# top: racialised rate, line + Wilson CI band
pct = social_aligned["rate"] * 100
ax1.plot(x, pct, marker="o", color="#C44E52", linewidth=2)
ax1.fill_between(x, social_aligned["ci_lo"] * 100, social_aligned["ci_hi"] * 100, color="#C44E52", alpha=0.15)
ax1.set_ylabel("racialised reviews,\nas % of social reviews")
print(f"N={int(yearly_social['n_reviews'].sum()):}")
# ax1.set_title(
#     f"Racialised rate and review volume by year (N={int(yearly_social['n_reviews'].sum()):,} social reviews",
    #f"pre-{CUTOFF_YEAR} pooled and 2026 partial year dropped)",
#     fontsize=16 , fontweight="bold"
# )


for spine in ["top", "right"]:
    ax1.spines[spine].set_visible(False)

# bottom: review volume, all reviews vs social = yes reviews
ax2.bar([i + width / 2 for i in x], social_aligned["n_reviews"], width=width, color="#C44E52", label="social = yes")
ax2.bar([i - width / 2 for i in x], yearly_totals["n_reviews"], width=width, color="#4C72B0", label="all reviews")
ax2.set_ylabel("reviews")
ax2.set_xlabel("review year")
ax2.set_xticks(x)
ax2.set_xticklabels(buckets, rotation=45, ha="right")
ax2.legend(fontsize=9)
for spine in ["top", "right"]:
    ax2.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_pre2016_bucketed.png", dpi=150, bbox_inches="tight")
plt.show()



We see that the racilaised rate has increased over time and particularly after 2021. Now we want to see if the racialised rate changes over seasons.
We would look at `summer` and `other seasons`which we could call `winter`

In [ ]:
# select only the actually-parsed (non-estimated) timestamps for the month/season analysis, since the
# estimated timestamps are only accurate to the day and we want to avoid any bias from the
ts_social_actual = corpus.loc[
    social_mask & corpus["review_timestamp_est"].notna() & ~corpus["timestamp_is_estimated"]
].copy()

ts_social_actual["review_year_est"] = ts_social_actual["review_timestamp_est"].dt.year
trend_window = ts_social_actual[ts_social_actual["review_year_est"].between(2016, 2025)].copy()
trend_window["year_c"] = trend_window["review_year_est"] - trend_window["review_year_est"].mean()

print("--- indoor/outdoor: trend within each season, separately ---")
for season in ["indoor", "outdoor"]:
    sub = trend_window[trend_window["pool_season"] == season]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{season:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int = smf.logit("is_racialised ~ year_c * C(pool_season)", data=trend_window).fit(disp=0)
inter_coef, inter_p = m_int.params["year_c:C(pool_season)[T.outdoor]"], m_int.pvalues["year_c:C(pool_season)[T.outdoor]"]
print(f"interaction (does outdoor's trend differ from indoor's?): coef={inter_coef:.4f}, p={inter_p:.2e}")

print("\n--- summer/other: trend within each group, separately ---")
for grp in ["Summer", "Other"]:
    sub = trend_window[trend_window["summer_vs_other"] == grp]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{grp:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int2 = smf.logit("is_racialised ~ year_c * C(summer_vs_other)", data=trend_window).fit(disp=0)
inter_coef2, inter_p2 = m_int2.params["year_c:C(summer_vs_other)[T.Summer]"], m_int2.pvalues["year_c:C(summer_vs_other)[T.Summer]"]
print(f"interaction (does Summer's trend differ from Other's?): coef={inter_coef2:.4f}, p={inter_p2:.2e}")


In [ ]:
def per_year_season_test(df, group_col, groups, year_col="review_year_est"):
    rows = []
    for yr, g in df.groupby(year_col):
        ct = pd.crosstab(g[group_col], g["is_racialised"])
        if ct.shape != (2, 2):
            continue
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        n = ct.values.sum()
        v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        rate_a = ct.loc[groups[0], 1] / ct.loc[groups[0]].sum() * 100
        rate_b = ct.loc[groups[1], 1] / ct.loc[groups[1]].sum() * 100
        rows.append({
            "year": yr, f"{groups[0]}_rate": rate_a, f"{groups[1]}_rate": rate_b,
            f"n_{groups[0]}": ct.loc[groups[0]].sum(), f"n_{groups[1]}": ct.loc[groups[1]].sum(),
            "chi2": chi2, "p": p, "V": v,
        })
    return pd.DataFrame(rows)

so_yearly = per_year_season_test(trend_window, "summer_vs_other", ["Summer", "Other"])
print("\n\nSummer vs. Other, tested separately within each year:")
print(so_yearly.round(4).to_string(index=False))
print(f"\nyears with a significant (p<0.05) within-year Summer/Other gap: {so_yearly.loc[so_yearly['p']<0.05,'year'].tolist()}")

fig1, ax1 = plt.subplots(figsize=(12, 12))

# 2. Are there significant connections betwen the racialised rate and socio economic and political factors

In [ ]:
# creaeing constituenccy level aggregation
# consistent naming with the shapefile/join in section 1)
backbone = socio_political[["state", "constituency_code", "constituency"]].copy()

agg = (
    corpus.groupby("constituency_code")
    .agg(
        pool_count=("unique_identifier", "nunique"),
        total_text_reviews=("review_id", "count"),
        social_reviews=("social", lambda s: (s == "yes").sum()),
        racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
    )
    .reset_index()
)


const_agg = backbone.merge(agg, on="constituency_code", how="left")
for col in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[col] = const_agg[col].fillna(0).astype(int)

# pools that ever got a social review -- the correct denominator for "share of pools with a
# racialised review", since racialised reviews are a subset of social ones (a pool with zero
# social reviews can never register as racialised).
social_pools = (
    corpus.loc[corpus["social"] == "yes"]
    .groupby("constituency_code")["unique_identifier"].nunique()
    .rename("pools_with_social")
)
const_agg = const_agg.merge(social_pools, on="constituency_code", how="left")
const_agg["pools_with_social"] = const_agg["pools_with_social"].fillna(0).astype(int)

n_zero_social_pool = (const_agg["pools_with_social"] == 0).sum()
n_zero_racialised = (const_agg["racialised_reviews"] == 0).sum()
print(f"constituencies with zero pools: {n_zero_social_pool} / 299")
print(f"constituencies with zero racialised reviews: {n_zero_racialised} / 299")
print()
print("pool_count per constituency:")
print(const_agg["pool_count"].describe())
print()
print("total_text_reviews per constituency:")
print(const_agg["total_text_reviews"].describe())    

For the racialsied rate we consider constituencies with >= 10 racialised reviews

In [ ]:
const_agg_reliable = const_agg.loc[const_agg["racialised_reviews"] >=10].copy()
const_agg["racialised_rate"] = const_agg['racialised_reviews'] / const_agg['social_reviews']
